# Refua Multi-Ligand Notebook: DHFR + Methotrexate + NADPH

> **Audience:** beginner to advanced.  
> **Goal:** build a single Refua `Complex` with **one target and two ligands**, then run ligand-specific affinity requests in a reproducible way.

## Why this target/ligand set is interesting
- **Target:** *E. coli* dihydrofolate reductase (DHFR), a classic enzyme used in mechanistic and inhibitor studies.
- **Ligand 1:** **methotrexate (MTX)**, a canonical DHFR inhibitor scaffold.
- **Ligand 2:** **NADPH/NADP cofactor analog context (NAP)**, representing the enzyme cofactor side of the active-site chemistry.
- **Key structural precedent:** PDB **1RH3** captures DHFR with **both MTX and NAP bound** in one crystal structure, making this a natural multi-ligand showcase.

## What you will do
1. Define one protein target and two ligand objects from web-sourced structural descriptors.
2. Build one `Complex([target, ligand_1, ligand_2])`.
3. Run two affinity-focused folds by explicitly selecting each ligand binder.
4. Compare outputs and learn the multi-ligand interpretation pattern.

## Important multi-ligand API note
When a complex has more than one ligand, affinity must be requested with an explicit binder (`request_affinity(ligand_obj)`), otherwise binder selection is ambiguous.


In [ ]:
%load_ext refua_notebook
import refua_notebook

refua_notebook.activate()


In [ ]:
from IPython.display import display
from refua import Complex, Protein, SM


## Define target and ligands

This uses:
- DHFR sequence from **RCSB FASTA: 1RH3 chain A**
- MTX and NAP stereo SMILES from **RCSB chemical component data API** (`MTX`, `NAP`)

Using structure-linked descriptors keeps the example reproducible and directly tied to known experimental context.


In [ ]:
DHFR_SEQUENCE_1RH3 = (
    "MISLIAALAVDRVIGMENAMPWNLPADLAWFKRNTLDKPVIMGRHTWESIGRPLPGRKNIILSSQPGTDDRVTWVKSVDEAIAACGDVPEIMVIGGGRVYEQFLPKAQKLYLTHIDAEVEGDTHFPDYEPDDWESVFSEFHDADAQNSHSYCFEILERR"
)

# RCSB chemcomp MTX (smilesstereo).
MTX_SMILES = "CN(Cc1cnc2c(n1)c(nc(n2)N)N)c3ccc(cc3)C(=O)N[C@@H](CCC(=O)O)C(=O)O"

# RCSB chemcomp NAP (smilesstereo).
NAP_SMILES = "c1cc(c[n+](c1)[C@H]2[C@@H]([C@@H]([C@H](O2)CO[P@@](=O)([O-])O[P@](=O)(O)OC[C@@H]3[C@H]([C@H]([C@@H](O3)n4cnc5c4ncnc5N)OP(=O)(O)O)O)O)O)C(=O)N"

target = Protein(DHFR_SEQUENCE_1RH3, ids="A")
ligand_mtx = SM(MTX_SMILES)
ligand_nap = SM(NAP_SMILES)


## Quick input sanity check

Display all three objects before folding.

This catches common setup problems early:
- malformed sequence strings
- invalid ligand stereochemistry/valence encodings
- unexpected molecule identity


In [ ]:
display(target)
display(ligand_mtx)
display(ligand_nap)


## Build one multi-ligand complex

This is the core showcase step: one target with two ligands in a single `Complex`.

In this setup, Refua assigns one chain ID per entity (typically `A` for protein, then `B`/`C` for ligands by order).

We intentionally keep one shared `complex_spec` so both ligand-specific affinity runs use the same structural context.


In [ ]:
complex_spec = Complex(
    [target, ligand_mtx, ligand_nap],
    name="dhfr_mtx_nap_multiligand",
)

complex_spec


## Fold with affinity focused on ligand 1 (MTX)

Because there are two ligands, we pass an explicit binder object: `request_affinity(ligand_mtx)`.

Interpret this as an MTX-centered ranking signal under the shared multi-ligand context.


In [ ]:
result_mtx = complex_spec.request_affinity(ligand_mtx).fold()

result_mtx.affinity


## Fold with affinity focused on ligand 2 (NAP)

Run the same complex again, but switch affinity focus to `ligand_nap`.

This gives a second ligand-centered signal without rebuilding the system definition.


In [ ]:
result_nap = complex_spec.request_affinity(ligand_nap).fold()

result_nap.affinity


## How to interpret and what to do next

Practical interpretation guardrails:
- Treat affinity values as **relative prioritization signals**, not standalone experimental truth.
- In multi-ligand systems, keep binder selection explicit in every affinity call.
- Compare ligand-focused runs together with structural plausibility before making decisions.

Natural follow-ups:
1. Create a branch that mutates the target (e.g., resistance-associated variants) and rerun both ligand-focused affinity requests.
2. Swap one ligand for an analog and compare shifts in the two-ligand context.

## Science references

- RCSB PDB 1RH3 (DHFR complexed with MTX and reduced-form NADP/NAP): https://www.rcsb.org/structure/1RH3
- RCSB FASTA 1RH3 (chain A sequence used here): https://www.rcsb.org/fasta/entry/1RH3/display
- RCSB chemcomp MTX descriptor API (SMILES source): https://data.rcsb.org/rest/v1/core/chemcomp/MTX
- RCSB chemcomp NAP descriptor API (SMILES source): https://data.rcsb.org/rest/v1/core/chemcomp/NAP
- Sawaya & Kraut, *Biochemistry* (1997), ecDHFR loop/subdomain movements (PMID: 9012674): https://pubmed.ncbi.nlm.nih.gov/9012674/
- FDA preclinical research basics (why computational hypotheses need experimental validation): https://www.fda.gov/patients/drug-development-process/step-2-preclinical-research
